In [1]:
!pip install torch "transformers>=5" huggingface_hub psutil ipykernel

   ---------------------------------------- 0.0/124.1 MB ? eta -:--:--
   ---------------------------------------- 0.0/124.1 MB 1.9 MB/s eta 0:01:05
   ---------------------------------------- 0.0/124.1 MB 1.9 MB/s eta 0:01:05
   ---------------------------------------- 0.1/124.1 MB 581.0 kB/s eta 0:03:34
   ---------------------------------------- 0.2/124.1 MB 871.5 kB/s eta 0:02:23
   ---------------------------------------- 0.2/124.1 MB 871.5 kB/s eta 0:02:23
   ---------------------------------------- 0.3/124.1 MB 874.6 kB/s eta 0:02:22
   ---------------------------------------- 0.4/124.1 MB 1.3 MB/s eta 0:01:37
   ---------------------------------------- 0.5/124.1 MB 1.2 MB/s eta 0:01:41
   ---------------------------------------- 0.9/124.1 MB 1.9 MB/s eta 0:01:06
   ---------------------------------------- 1.0/124.1 MB 2.0 MB/s eta 0:01:03
   ---------------------------------------- 1.3/124.1 MB 2.3 MB/s eta 0:00:53
    --------------------------------------- 1.6/124.1 MB 2.6 MB

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
llama-index-indices-managed-llama-cloud 0.2.7 requires llama-index-core<0.11.0,>=0.10.48.post1, but you have llama-index-core 0.14.23 which is incompatible.
llama-index-llms-openai 0.1.31 requires llama-index-core<0.11.0,>=0.10.57, but you have llama-index-core 0.14.23 which is incompatible.
llama-index-multi-modal-llms-openai 0.1.9 requires llama-index-core<0.11.0,>=0.10.1, but you have llama-index-core 0.14.23 which is incompatible.
llama-index-program-openai 0.1.7 requires llama-index-core<0.11.0,>=0.10.57, but you have llama-index-core 0.14.23 which is incompatible.
llama-index-question-gen-openai 0.1.3 requires llama-index-core<0.11.0,>=0.10.1, but you have llama-index-core 0.14.23 which is incompatible.
llama-index-readers-file 0.1.33 requires llama-index-core<0.11.0,>=0.10.37.post1, but you have llama-index

In [15]:
from pathlib import Path
import torch

MODEL_ID = "Qwen/Qwen2.5-0.5B-Instruct"
RESULTS_DIR = Path("results")
RESULTS_DIR.mkdir(exist_ok=True)

GEN_DTYPE = torch.bfloat16 
MAX_NEW_TOKENS = 64
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("device", DEVICE)


device cpu


In [3]:
import os, sys, platform, time, json, gc, subprocess
from contextlib import contextmanager
from datetime import datetime, timezone
import psutil, transformers, huggingface_hub

def env_info():
    info = {
        "python": sys.version.split()[0],
        "platform": platform.platform(),
        "cpu": platform.processor() or platform.machine(),
        "cpu_logical_cores": psutil.cpu_count(logical=True),
        "ram_total_gb": round(psutil.virtual_memory().total / 1e9, 1),
        "torch": torch.__version__,
        "transformers": transformers.__version__,
        "huggingface_hub": huggingface_hub.__version__,
        "torch_threads": torch.get_num_threads(),
        "DEVICE": DEVICE,
    }
    
    if torch.cuda.is_available():
        info["gpu"] = torch.cuda.get_device_name(0)
    return info

ENV = env_info()
ENV


c:\Users\Lenovo\AppData\Local\Programs\Python\Python311\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


{'python': '3.11.9',
 'platform': 'Windows-10-10.0.26200-SP0',
 'cpu': 'Intel64 Family 6 Model 186 Stepping 2, GenuineIntel',
 'cpu_logical_cores': 16,
 'ram_total_gb': 16.9,
 'torch': '2.14.1+cpu',
 'transformers': '5.19.0',
 'huggingface_hub': '2.2.0',
 'torch_threads': 10,
 'DEVICE': 'cpu'}

In [4]:
from transformers import AutoConfig, AutoModelForCausalLM

@contextmanager
def timer():
    t = {}
    start = time.perf_counter()
    yield t
    t["s"] = time.perf_counter() - start
    
def rss_mb():
    return psutil.Process().memory_info().rss / 2**20

def save_result(name, data):
    record = {
        "name" : name,
        "time_utc" : datetime.now(timezone.utc).isoformat(),
        "model" : MODEL_ID,
        "env" : ENV, **data
    }
    
    with open(RESULTS_DIR / "hugging_face.jsonl", "a", encoding="utf-8") as f:
        f.write(json.dumps(record) + "\n")
    print(f"saved: {name}")
    
def count_unique_params(model):
    # model.parameters() already skips shared tensors, so a tied embedding is counted once
    return sum(p.numel() for p in model.parameters())

def estimate_memory(config, bytes_per_values=2, context_tokens=4096, num_requests=1):
    with torch.device("meta"):
        m = AutoModelForCausalLM.from_config(config)
    n_params = count_unique_params(m)
    
    del m
    
    head_dim = getattr(config, "head_dim", None) or config.hidden_size//config.num_attention_heads
    kv_heads = getattr(config, "num_key_value_heads", None) or config.num_attention_heads
    kv_per_token = 2 * config.num_hidden_layers * kv_heads * head_dim *bytes_per_values # K and V
    return {
        "params" : n_params,
        "head_dim": head_dim,
        "weight_bytes": n_params * bytes_per_values,
        "kv_bytes_per_token": kv_per_token,
        "kv_bytes_total": kv_per_token * context_tokens * num_requests,
    }
    
def gib(x):
    return x / 2**30

def mib(x):
    return x / 2**20

print("helpers ready")


helpers ready


In [5]:
config = AutoConfig.from_pretrained(MODEL_ID)

fields = [
    "model_type", "hidden_size", "intermediate_size", "num_hidden_layers", "num_attention_heads",
    "num_key_value_heads", "vocab_size", "max_position_embeddings", "tie_word_embeddings", "eos_token_id", "bos_token_id"
]

for f in fields:
    print(f"{f:26s} {getattr(config, f, None)}")
print(f"{'saved dtype':26s} {getattr(config, 'dtype', None) or getattr(config, 'torch_dtype', None)}")
print(f"{'query heads per KV head':26s} {config.num_attention_heads // config.num_key_value_heads}")


model_type                 qwen2
hidden_size                896
intermediate_size          4864
num_hidden_layers          24
num_attention_heads        14
num_key_value_heads        2
vocab_size                 151936
max_position_embeddings    32768
tie_word_embeddings        True
eos_token_id               151645
bos_token_id               151643
saved dtype                torch.bfloat16
query heads per KV head    7


In [6]:
est = estimate_memory(config, bytes_per_values=2)
print(f"parameters        : {est['params']:,}")
print(f"head_dim          : {est['head_dim']}")
print(f"KV per token (bf16): {est['kv_bytes_per_token']:,} bytes")

if MODEL_ID == "Qwen/Qwen2.5-0.5B-Instruct":
    assert est["params"] == 494_032_768
    assert est["head_dim"] == 64
    assert est["kv_bytes_per_token"] == 12_288
    print("✓ calculator matches the expected numbers")
    
# embedding share (and what tying saves)
emb = config.vocab_size * config.hidden_size
print(f"\nembedding matrix  : {emb:,} params ({emb / est['params']:.1%} of the model)")
print(f"tie_word_embeddings={config.tie_word_embeddings} -> lm_head reuses it; untied would add {emb:,} params")

print("\nWeight memory by precision:")
for name, b in [("fp32", 4), ("bf16/fp16", 2), ("int8", 1), ("4-bit", 0.5)]:
    print(f"  {name:10s} {est['params'] * b / 1e9:6.2f} GB")

print("\nKV cache (bf16):")
for ctx in [4096, 32768]:
    for reqs in [1, 32]:
        kv = estimate_memory(config, 2, ctx, reqs)["kv_bytes_total"]
        print(f"  context {ctx:>6} × {reqs:>2} requests = {mib(kv):8.0f} MiB ({gib(kv):5.2f} GiB)")



parameters        : 494,032,768
head_dim          : 64
KV per token (bf16): 12,288 bytes
✓ calculator matches the expected numbers

embedding matrix  : 136,134,656 params (27.6% of the model)
tie_word_embeddings=True -> lm_head reuses it; untied would add 136,134,656 params

Weight memory by precision:
  fp32         1.98 GB
  bf16/fp16    0.99 GB
  int8         0.49 GB
  4-bit        0.25 GB

KV cache (bf16):
  context   4096 ×  1 requests =       48 MiB ( 0.05 GiB)
  context   4096 × 32 requests =     1536 MiB ( 1.50 GiB)
  context  32768 ×  1 requests =      384 MiB ( 0.38 GiB)
  context  32768 × 32 requests =    12288 MiB (12.00 GiB)


In [7]:
from huggingface_hub import snapshot_download

if os.path.isdir(MODEL_ID):
    local_path = MODEL_ID
    print("using local folder: ", local_path)
    
else:
    with timer() as t:
        local_path = snapshot_download(MODEL_ID)
    print(f"download/cache check took {t['s']:.1f}s  (fast if already cached)")
    
for p in sorted(Path(local_path).iterdir()):
    if p.is_file():
        print(f"{p.stat().st_size:>14,} bytes  {p.name}")

st = next(Path(local_path).glob("*.safetensors"), None)
if st is not None:
    print(f"\nsafetensors file minus bf16 weights = {st.stat().st_size - est['params'] * 2:,} bytes (the JSON header)")

Fetching 10 files: 100%|██████████| 10/10 [00:26<00:00,  2.64s/it]

download/cache check took 28.5s  (fast if already cached)
         1,519 bytes  .gitattributes
           659 bytes  config.json
           242 bytes  generation_config.json
        11,343 bytes  LICENSE
     1,671,839 bytes  merges.txt
   988,097,824 bytes  model.safetensors
         4,917 bytes  README.md
     7,031,645 bytes  tokenizer.json
         7,305 bytes  tokenizer_config.json
     2,776,833 bytes  vocab.json

safetensors file minus bf16 weights = 32,288 bytes (the JSON header)


In [8]:
LOAD_SCRIPT = r"""
import sys, json, time, psutil, torch
from transformers import AutoModelForCausalLM
path, dtype_name = sys.argv[1], sys.argv[2]
proc = psutil.Process()
before = proc.memory_info().rss / 2**20
t0 = time.perf_counter()
model = AutoModelForCausalLM.from_pretrained(path, dtype=getattr(torch, dtype_name))
load_s = time.perf_counter() - t0
after = proc.memory_info().rss / 2**20
n = sum(p.numel() for p in model.parameters())
tied = model.get_output_embeddings().weight is model.get_input_embeddings().weight
print(json.dumps({"dtype": dtype_name, "load_s": load_s, "rss_before_mb": before, "rss_after_mb": after,
                  "ram_added_mb": after - before, "params": n, "loaded_dtype": str(next(model.parameters()).dtype),
                  "lm_head_tied": tied}))
"""

def measure_load(dtype_name):
    r = subprocess.run([sys.executable, "-c", LOAD_SCRIPT, str(local_path), dtype_name],
                       capture_output=True, text=True)
    if r.returncode != 0:
        print(r.stderr[-3000:])
        raise RuntimeError("load failed")
    return json.loads(r.stdout.strip().splitlines()[-1])

load_results = {}
for dt in ["bfloat16", "float32"]:
    res = measure_load(dt)
    load_results[dt] = res
    bytes_per = 2 if dt == "bfloat16" else 4
    expected_mb = est["params"] * bytes_per / 2**20
    print(f"{dt:9s} load {res['load_s']:5.1f}s | RAM added {res['ram_added_mb']:7.0f} MiB "
          f"(weights alone: {expected_mb:6.0f} MiB) | params {res['params']:,} | tied={res['lm_head_tied']}")
    save_result(f"load_{dt}", res)

Exception in thread Thread-8 (_readerthread):
Traceback (most recent call last):
  File "c:\Users\Lenovo\AppData\Local\Programs\Python\Python311\Lib\threading.py", line 1045, in _bootstrap_inner
    self.run()
  File "C:\Users\Lenovo\AppData\Roaming\Python\Python311\site-packages\ipykernel\ipkernel.py", line 772, in run_closure
    _threading_Thread_run(self)
  File "c:\Users\Lenovo\AppData\Local\Programs\Python\Python311\Lib\threading.py", line 982, in run
    self._target(*self._args, **self._kwargs)
  File "c:\Users\Lenovo\AppData\Local\Programs\Python\Python311\Lib\subprocess.py", line 1599, in _readerthread
    buffer.append(fh.read())
                  ^^^^^^^^^
  File "c:\Users\Lenovo\AppData\Local\Programs\Python\Python311\Lib\encodings\cp1252.py", line 23, in decode
    return codecs.charmap_decode(input,self.errors,decoding_table)[0]
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
UnicodeDecodeError: 'charmap' codec can't decode byte 0x8f in position 82: ch

bfloat16  load   4.5s | RAM added     956 MiB (weights alone:    942 MiB) | params 494,032,768 | tied=True
saved: load_bfloat16


Exception in thread Thread-10 (_readerthread):
Traceback (most recent call last):
  File "c:\Users\Lenovo\AppData\Local\Programs\Python\Python311\Lib\threading.py", line 1045, in _bootstrap_inner
    self.run()
  File "C:\Users\Lenovo\AppData\Roaming\Python\Python311\site-packages\ipykernel\ipkernel.py", line 772, in run_closure
    _threading_Thread_run(self)
  File "c:\Users\Lenovo\AppData\Local\Programs\Python\Python311\Lib\threading.py", line 982, in run
    self._target(*self._args, **self._kwargs)
  File "c:\Users\Lenovo\AppData\Local\Programs\Python\Python311\Lib\subprocess.py", line 1599, in _readerthread
    buffer.append(fh.read())
                  ^^^^^^^^^
  File "c:\Users\Lenovo\AppData\Local\Programs\Python\Python311\Lib\encodings\cp1252.py", line 23, in decode
    return codecs.charmap_decode(input,self.errors,decoding_table)[0]
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
UnicodeDecodeError: 'charmap' codec can't decode byte 0x8f in position 299: 

float32   load   8.2s | RAM added    1588 MiB (weights alone:   1885 MiB) | params 494,032,768 | tied=True
saved: load_float32


In [16]:
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained(local_path)
model = AutoModelForCausalLM.from_pretrained(local_path, dtype=GEN_DTYPE).to(DEVICE).eval()
print("loaded:", next(model.parameters()).dtype, "on", DEVICE)


Loading weights: 100%|██████████| 290/290 [00:00<00:00, 442.20it/s]

loaded: torch.bfloat16 on cpu


In [17]:
print("tokenizer class :", type(tokenizer).__name__)
print("vocab (tokenizer):", len(tokenizer), "| vocab_size (config, embedding rows):", config.vocab_size)

for name in ["bos_token", "eos_token", "pad_token", "unk_token"]:
    tok = getattr(tokenizer, name)
    tid = getattr(tokenizer, name + "_id")
    print(f"{name:10s} {tok!r:20s} id={tid}")
print("\nall special tokens:", tokenizer.all_special_tokens)


tokenizer class : Qwen2Tokenizer
vocab (tokenizer): 151665 | vocab_size (config, embedding rows): 151936
bos_token  None                 id=None
eos_token  '<|im_end|>'         id=151645
pad_token  '<|endoftext|>'      id=151643
unk_token  None                 id=None

all special tokens: ['<|im_end|>', '<|endoftext|>', '<|im_start|>', '<|object_ref_start|>', '<|object_ref_end|>', '<|box_start|>', '<|box_end|>', '<|quad_start|>', '<|quad_end|>', '<|vision_start|>', '<|vision_end|>', '<|vision_pad|>', '<|image_pad|>', '<|video_pad|>']


In [18]:
samples = {
    "english": "The weather is very nice today.",
    "tamil":   "இன்று வானிலை மிகவும் நன்றாக உள்ளது.",
    "code":    "def add(a, b):\n    return a + b",
}

token_counts = {}
for name, text in samples.items():
    ids = tokenizer(text, add_special_tokens=False)["input_ids"]
    token_counts[name] = len(ids)
    print(f"{name:8s} chars={len(text):3d} tokens={len(ids):3d} tokens/char={len(ids)/len(text):.2f}")
    print("         ", tokenizer.convert_ids_to_tokens(ids)[:15])
    print("          round-trip decode exact:", tokenizer.decode(ids) == text)
    
save_result("token_counts", {"samples": samples, "token_counts": token_counts})

english  chars= 31 tokens=  7 tokens/char=0.23
          ['The', 'Ġweather', 'Ġis', 'Ġvery', 'Ġnice', 'Ġtoday', '.']
          round-trip decode exact: True
tamil    chars= 35 tokens= 38 tokens/char=1.09
          ['à®ĩ', 'à®©', 'à¯įà®', '±', 'à¯ģ', 'Ġà®', 'µ', 'à®', '¾', 'à®©', 'à®¿à®', '²', 'à¯', 'Ī', 'Ġà®']
          round-trip decode exact: True
code     chars= 31 tokens= 11 tokens/char=0.35
          ['def', 'Ġadd', '(a', ',', 'Ġb', '):Ċ', 'ĠĠĠ', 'Ġreturn', 'Ġa', 'Ġ+', 'Ġb']
          round-trip decode exact: True
saved: token_counts


In [19]:
no_system = [{"role": "user", "content": "Hello"}]
with_system = [{"role": "system", "content": "You are a concise assistant."},
               {"role": "user", "content": "Hello"}]

for label, msgs in [("WITHOUT system message", no_system), ("WITH system message", with_system)]:
    text = tokenizer.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True)
    n = len(tokenizer(text, add_special_tokens=False)["input_ids"])
    print(f"===== {label} ({n} tokens) =====")
    print(text)

no_gen = tokenizer.apply_chat_template(no_system, tokenize=False, add_generation_prompt=False)
print("===== add_generation_prompt=False (last 40 chars) =====")
print(repr(no_gen[-40:]))

===== WITHOUT system message (30 tokens) =====
<|im_start|>system
You are Qwen, created by Alibaba Cloud. You are a helpful assistant.<|im_end|>
<|im_start|>user
Hello<|im_end|>
<|im_start|>assistant

===== WITH system message (20 tokens) =====
<|im_start|>system
You are a concise assistant.<|im_end|>
<|im_start|>user
Hello<|im_end|>
<|im_start|>assistant

===== add_generation_prompt=False (last 40 chars) =====
'_end|>\n<|im_start|>user\nHello<|im_end|>\n'


In [20]:
print(model.generation_config)

GenerationConfig {
  "bos_token_id": 151643,
  "do_sample": true,
  "eos_token_id": [
    151645,
    151643
  ],
  "pad_token_id": 151643,
  "repetition_penalty": 1.1,
  "temperature": 0.7,
  "top_k": 20,
  "top_p": 0.8
}



In [21]:
messages = [{"role": "user", "content": "Explain what a KV cache is in two sentences."}]
prompt_text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
inputs = tokenizer(prompt_text, return_tensors="pt", add_special_tokens=False).to(DEVICE)
prompt_len = inputs["input_ids"].shape[1]
print("prompt tokens:", prompt_len)

GREEDY = dict(do_sample=False, temperature=None, top_p=None, top_k=None, repetition_penalty=1.0)

@torch.inference_mode()
def run(gen_kwargs):
    if DEVICE == "cuda":
        torch.cuda.synchronize()
    t0 = time.perf_counter()
    out = model.generate(**inputs, max_new_tokens=MAX_NEW_TOKENS, **gen_kwargs)
    
    if DEVICE == "cuda":
        torch.cuda.synchronize()
        
    secs = time.perf_counter() - t0
    new_ids = out[0, prompt_len:]
    return {
        "text": tokenizer.decode(new_ids, skip_special_tokens=True),
        "new_tokens": int(new_ids.shape[0]), 
        "seconds": secs,
        "tok_per_s": new_ids.shape[0] / secs
    }
    
_ = run(GREEDY)

print("\n===== GREEDY (do_sample=False) x3 =====")
greedy_runs = [run(GREEDY) for _ in range(3)]
for r in greedy_runs:
    print(f"{r['new_tokens']:3d} tokens  {r['seconds']:5.2f}s  {r['tok_per_s']:6.1f} tok/s | {r['text'][:70]!r}")
print("identical:", len({r["text"] for r in greedy_runs}) == 1)

print("\n===== DEFAULT generation_config (sampling) x3 =====")
sample_runs = [run({}) for _ in range(3)]
for r in sample_runs:
    print(f"{r['new_tokens']:3d} tokens  {r['seconds']:5.2f}s  {r['tok_per_s']:6.1f} tok/s | {r['text'][:70]!r}")
print("all identical:", len({r["text"] for r in sample_runs}) == 1)

save_result("generation", {"gen_dtype": str(GEN_DTYPE), "prompt_tokens": prompt_len,
                           "max_new_tokens": MAX_NEW_TOKENS,
                           "greedy": greedy_runs, "sampling": sample_runs})

prompt tokens: 40

===== GREEDY (do_sample=False) x3 =====
 51 tokens   3.15s    16.2 tok/s | 'A KVS (Key-Value Store) is a type of cache that stores data in a datab'
 51 tokens   3.23s    15.8 tok/s | 'A KVS (Key-Value Store) is a type of cache that stores data in a datab'
 51 tokens   3.12s    16.4 tok/s | 'A KVS (Key-Value Store) is a type of cache that stores data in a datab'
identical: True

===== DEFAULT generation_config (sampling) x3 =====
 42 tokens   3.27s    12.8 tok/s | 'A KVD (Key-Value) cache stores data with keys and values as key-value '
 46 tokens   3.48s    13.2 tok/s | 'A KVS (Key-Value Store) cache is a data structure that stores key-valu'
 39 tokens   3.26s    11.9 tok/s | 'A K-V cache is a data structure used to store key-value pairs of data,'
all identical: False
saved: generation
